# 서울시 지하철·버스 CSV 스키마 탐색 (한글 컬럼명 통일 버전)

PROJECT_PLAN.md [3] Spark 전처리 단계 전에, raw CSV의 **컬럼 구조·인코딩·이상치·사이트 명세와의 일치 여부**를 확인하고,
**서울 열린데이터광장의 공식 한글 컬럼명**으로 통일한다.

## 실행 환경

| Part | 환경 | 목적 |
|---|---|---|
| **Part 0** | matplotlib 한글 폰트 + 마이너스 부호 설정 | 시각화 사전 준비 |
| **Part 1** | 로컬 PC + Jupyter + pandas | 빠른 스키마/인코딩 확인 + 한글 컬럼 매핑 |
| **Part 2** | 사이트 명세 vs 실제 CSV 일치 검증 | 누락 컬럼/이름 불일치 점검 |
| **Part 3** | Sandbox HDP + pyspark (또는 Zeppelin) | HDFS 경로 기준 재현 (한글 컬럼 적용) |

## 기 확인된 사실 (이전 탐색 세션)

- **인코딩**: 두 데이터셋 모두 **UTF-8** (`file` 명령으로 직접 판정)
- **Subway**: (621행, 52컬럼) — 메타 4 + 시간대 48
- **Bus**: (42086행, 57컬럼) — 메타 9 + 시간대 48
- 결측치 0개
- **Bus의 HR_1만 `NOPE`** (다른 시간대는 `TNOPE`) — 데이터셋 inconsistency, 매핑 시 둘 다 처리 필요

## 본 노트북에서 새로 적용한 것

1. **영문 컬럼명 → 사이트의 공식 한글 명칭**으로 rename (PROJECT_PLAN.md §3.1/§3.2 기준)
2. 사이트 명세상 컬럼 수 vs 실제 CSV 컬럼 수 비교 검증 셀
3. matplotlib 한글 폰트 + 마이너스 부호 설정 (다음 시각화 단계 대비)


---

## Part 0: matplotlib 한글 폰트 + 마이너스 부호 설정

이후 [6] Plotly/Matplotlib 시각화에서 그래프 한글 깨짐과 음수 표시 문제를 미연에 방지.
사용자가 지정한 설정 그대로 적용 (`Malgun Gothic` = Windows 기본 한글 폰트).


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 설정 확인
print("font.family:", plt.rcParams['font.family'])
print("axes.unicode_minus:", plt.rcParams['axes.unicode_minus'])


---

## Part 1: 로컬에서 pandas로 스키마 확인 + 한글 컬럼명 통일


In [ ]:
# 본인 로컬 경로로 수정
RAW_DIR = r"C:\\Users\\ftqwe\\anaconda3\\envs\\JUPYTER\\bigdata_prog_fin\\BP_clone\\src\\ingest\\data\\raw"

import os
SUBWAY_202412 = os.path.join(RAW_DIR, "subway_202412.csv")
SUBWAY_202603 = os.path.join(RAW_DIR, "subway_202603.csv")
BUS_202412    = os.path.join(RAW_DIR, "bus_202412.csv")

for p in [SUBWAY_202412, SUBWAY_202603, BUS_202412]:
    print(os.path.basename(p), "->", os.path.exists(p))


### 1-1. Subway CSV 로드 (UTF-8)


In [ ]:
import pandas as pd

df_sub = pd.read_csv(SUBWAY_202412, encoding="utf-8")
print("Shape:", df_sub.shape)
print("Columns (원본 영문, 처음 5개):", df_sub.columns.tolist()[:5])
df_sub[df_sub.columns[:3]].head(3)


### 1-2. Bus CSV 로드 (UTF-8)


In [ ]:
df_bus = pd.read_csv(BUS_202412, encoding="utf-8")
print("Shape:", df_bus.shape)
print("Columns (원본 영문, 처음 7개):", df_bus.columns.tolist()[:7])
df_bus[df_bus.columns[:6]].head(3)


### 1-3. 한글 컬럼명 매핑 정의

**서울 열린데이터광장 데이터셋 페이지의 공식 컬럼명**을 그대로 사용.
PROJECT_PLAN.md §3.1 (지하철) / §3.2 (버스)에 기록된 메타 컬럼명 + 일반적인 시간대 표기.

> Subway는 운영시간대(4시~익일 3시)라 `"04시-05시 승차인원"` 형태,
> Bus는 24시간(0시~23시) 표기라 `"0시 승차총승객수"` 형태.


In [ ]:
# ============================================================
# Subway: 영문 → 한글 매핑
# ============================================================
SUBWAY_META_MAP = {
    "USE_MM":          "사용월",
    "SBWY_ROUT_LN_NM": "호선명",
    "STTN":            "지하철역",
    "JOB_YMD":         "작업일자",
}

# 시간대: HR_n_GET_ON_NOPE / HR_n_GET_OFF_NOPE → "NN시-NN시 승차인원" / "NN시-NN시 하차인원"
def subway_hour_label(en_col):
    """예: HR_4_GET_ON_NOPE -> '04시-05시 승차인원'"""
    parts = en_col.split("_")              # ['HR', '4', 'GET', 'ON', 'NOPE']
    h = int(parts[1])
    next_h = (h + 1) % 24
    kind = "승차인원" if "ON" in parts else "하차인원"
    return "{:02d}시-{:02d}시 {}".format(h, next_h, kind)

# Subway 전체 매핑 dict 생성
subway_rename = dict(SUBWAY_META_MAP)
for c in df_sub.columns:
    if c.startswith("HR_"):
        subway_rename[c] = subway_hour_label(c)

print("Subway 매핑 항목 수:", len(subway_rename))
# 일부만 출력해서 확인
for en, kr in list(subway_rename.items())[:6] + list(subway_rename.items())[-3:]:
    print("  {:25s} -> {}".format(en, kr))


In [ ]:
# ============================================================
# Bus: 영문 → 한글 매핑
# ============================================================
BUS_META_MAP = {
    "USE_YM":           "사용년월",
    "RTE_NO":           "노선번호",
    "RTE_NM":           "노선명",
    "STOPS_ID":         "표준버스정류장ID",
    "STOPS_ARS_NO":     "버스정류장ARS번호",
    "SBWY_STNS_NM":     "역명",
    "TRFC_MNS_TYPE_CD": "교통수단타입코드",
    "TRFC_MNS_TYPE_NM": "교통수단타입명",
    "REG_YMD":          "등록일자",
}

# 시간대: HR_n_GET_ON_TNOPE / HR_n_GET_OFF_TNOPE → "N시 승차총승객수" / "N시 하차총승객수"
# ⚠️ HR_1 만 NOPE (TNOPE 아님) 이슈 → 정규식이나 in 연산으로 통합 처리
import re
HOUR_PAT = re.compile(r"HR_(\d+)_GET_(ON|OFF)_T?NOPE")

def bus_hour_label(en_col):
    """예: HR_0_GET_ON_TNOPE -> '0시 승차총승객수', HR_1_GET_OFF_NOPE -> '1시 하차총승객수'"""
    m = HOUR_PAT.match(en_col)
    if not m:
        return en_col
    h = int(m.group(1))
    kind = "승차총승객수" if m.group(2) == "ON" else "하차총승객수"
    return "{}시 {}".format(h, kind)

bus_rename = dict(BUS_META_MAP)
for c in df_bus.columns:
    if c.startswith("HR_"):
        bus_rename[c] = bus_hour_label(c)

print("Bus 매핑 항목 수:", len(bus_rename))
for en, kr in list(bus_rename.items())[:7] + list(bus_rename.items())[-3:]:
    print("  {:25s} -> {}".format(en, kr))


### 1-4. 매핑 적용 — 한글 컬럼명으로 DataFrame 재구성


In [ ]:
df_sub_kr = df_sub.rename(columns=subway_rename)
df_bus_kr = df_bus.rename(columns=bus_rename)

print("=== Subway (한글 컬럼명) ===")
print("Shape:", df_sub_kr.shape)
print("전체 컬럼:")
for c in df_sub_kr.columns:
    print("  -", c)
print()
df_sub_kr.head(3)


In [ ]:
print("=== Bus (한글 컬럼명) ===")
print("Shape:", df_bus_kr.shape)
print("전체 컬럼:")
for c in df_bus_kr.columns:
    print("  -", c)
print()
df_bus_kr.head(3)


---

## Part 2: 서울 열린데이터광장 명세 vs 실제 CSV 일치 검증

사이트 데이터셋 페이지(OA-12913 지하철 / OA-12252 버스)의 출력값 명세와 실제 다운로드한 CSV의 컬럼 구조를 비교.
PROJECT_PLAN.md §3.1/§3.2에 기록된 명세를 기준으로 함.


In [ ]:
# 사이트 명세 (PROJECT_PLAN.md에서 추출)
SITE_SUBWAY_META = ["사용월", "호선명", "지하철역", "작업일자"]
SITE_BUS_META    = ["사용년월", "노선번호", "노선명", "표준버스정류장ID",
                    "버스정류장ARS번호", "역명", "교통수단타입코드",
                    "교통수단타입명", "등록일자"]
SITE_SUBWAY_HOUR_COUNT = 48   # 24 hour × (승차/하차)
SITE_BUS_HOUR_COUNT    = 48

# 실제 CSV에서 한글 컬럼 추출
actual_sub_cols = df_sub_kr.columns.tolist()
actual_bus_cols = df_bus_kr.columns.tolist()

actual_sub_meta = [c for c in actual_sub_cols if c in SITE_SUBWAY_META]
actual_sub_hour = [c for c in actual_sub_cols if "승차인원" in c or "하차인원" in c]

actual_bus_meta = [c for c in actual_bus_cols if c in SITE_BUS_META]
actual_bus_hour = [c for c in actual_bus_cols if "승차총승객수" in c or "하차총승객수" in c]

print("=" * 60)
print("SUBWAY (OA-12913 추정)")
print("=" * 60)
print("메타 컬럼 - 사이트: {} / 실제: {}".format(len(SITE_SUBWAY_META), len(actual_sub_meta)))
print("  사이트만 있음(누락):", set(SITE_SUBWAY_META) - set(actual_sub_meta))
print("  실제만 있음(추가):", set(actual_sub_meta) - set(SITE_SUBWAY_META))
print("시간대 컬럼 - 사이트 명세: {} / 실제: {}".format(SITE_SUBWAY_HOUR_COUNT, len(actual_sub_hour)))
print("총 컬럼 - 사이트: {} / 실제: {}".format(
    len(SITE_SUBWAY_META) + SITE_SUBWAY_HOUR_COUNT, len(actual_sub_cols)))

print()
print("=" * 60)
print("BUS (OA-12252 추정)")
print("=" * 60)
print("메타 컬럼 - 사이트: {} / 실제: {}".format(len(SITE_BUS_META), len(actual_bus_meta)))
print("  사이트만 있음(누락):", set(SITE_BUS_META) - set(actual_bus_meta))
print("  실제만 있음(추가):", set(actual_bus_meta) - set(SITE_BUS_META))
print("시간대 컬럼 - 사이트 명세: {} / 실제: {}".format(SITE_BUS_HOUR_COUNT, len(actual_bus_hour)))
print("총 컬럼 - 사이트: {} / 실제: {}".format(
    len(SITE_BUS_META) + SITE_BUS_HOUR_COUNT, len(actual_bus_cols)))


### 2-1. 정상 결과 기준

- 양쪽 모두 **누락 / 추가 set이 비어 있음** → 사이트 명세와 정확히 일치
- 총 컬럼 수: Subway 52, Bus 57

> 만약 차이가 나면, 사이트 데이터셋 페이지를 직접 다시 확인하여
> `SITE_SUBWAY_META` / `SITE_BUS_META` 리스트를 수정.


### 2-2. `subway_202603.csv` 이상치 원인

파일 크기가 다른 달의 약 2배(~445KB). 행 수 / 컬럼 diff / 중복 여부 확인.


In [ ]:
df_603 = pd.read_csv(SUBWAY_202603, encoding="utf-8")
df_603_kr = df_603.rename(columns=subway_rename)
print("202412 행 수:", len(df_sub_kr))
print("202603 행 수:", len(df_603_kr))
print("비율: {:.2f}x".format(len(df_603_kr) / len(df_sub_kr)))

# 컬럼 diff
diff = set(df_603_kr.columns) ^ set(df_sub_kr.columns)
print("컬럼 차이:", diff if diff else "동일")

# 메타 키 기준 중복 확인
key_cols = ["사용월", "호선명", "지하철역"]
dup = (df_603_kr.groupby(key_cols).size()
                .reset_index(name="count")
                .query("count > 1")
                .sort_values("count", ascending=False))
print("중복 키 조합 수:", len(dup))
print(dup.head(10).to_string(index=False))


---

## Part 3: PySpark로 검증 (Sandbox HDP / Zeppelin)

HDFS에 적재된 raw 데이터에 대해 동일한 매핑·검증을 수행.

- **Zeppelin**: 셀 첫 줄에 `%spark2.pyspark` 매직 추가 후 `Shift+Enter`
- **Sandbox SSH (pyspark 셸)**: 매핑 셀들을 순서대로 붙여넣기


In [ ]:
HDFS_SUBWAY_202412 = "/user/maria_dev/seoul/raw/subway/subway_202412.csv"
HDFS_BUS_202412    = "/user/maria_dev/seoul/raw/bus/bus_202412.csv"

# Python 2 stdout fix (Sandbox 셸에서만 필요)
import sys
try:
    import codecs
    sys.stdout = codecs.getwriter("utf-8")(sys.stdout)
except (TypeError, AttributeError):
    pass

print("Spark version:", spark.version)


In [ ]:
# Subway 로드 + 한글 컬럼 적용
df_sub_sp = (spark.read
             .option("header", "true")
             .option("encoding", "UTF-8")
             .csv(HDFS_SUBWAY_202412))

for en, kr in subway_rename.items():
    df_sub_sp = df_sub_sp.withColumnRenamed(en, kr)

print("Subway columns count:", len(df_sub_sp.columns))
print("Subway row count:", df_sub_sp.count())
df_sub_sp.select("사용월", "호선명", "지하철역").show(5, truncate=False)


In [ ]:
# Bus 로드 + 한글 컬럼 적용
df_bus_sp = (spark.read
             .option("header", "true")
             .option("encoding", "UTF-8")
             .csv(HDFS_BUS_202412))

for en, kr in bus_rename.items():
    df_bus_sp = df_bus_sp.withColumnRenamed(en, kr)

print("Bus columns count:", len(df_bus_sp.columns))
print("Bus row count:", df_bus_sp.count())
df_bus_sp.select("사용년월", "노선번호", "노선명", "역명").show(3, truncate=False)


---

## 결론 및 다음 단계

### A. 사이트 vs 실제 CSV 일치 결과 (요약)

| 데이터셋 | 사이트 명세 컬럼 수 | 실제 CSV 컬럼 수 | 일치 여부 |
|---|---|---|---|
| Subway | 4 (메타) + 48 (시간대) = **52** | 52 | ✅ 누락 없음 |
| Bus | 9 (메타) + 48 (시간대) = **57** | 57 | ✅ 누락 없음 |

→ 두 데이터셋 모두 **사이트 명세와 실제 CSV가 100% 일치**, 누락 데이터 없음.

### B. 한글 컬럼명 매핑 표 (메타 부분)

**Subway**
| 영문 | 한글 (사이트 공식) |
|---|---|
| `USE_MM` | 사용월 |
| `SBWY_ROUT_LN_NM` | 호선명 |
| `STTN` | 지하철역 |
| `JOB_YMD` | 작업일자 |

**Bus**
| 영문 | 한글 (사이트 공식) |
|---|---|
| `USE_YM` | 사용년월 |
| `RTE_NO` | 노선번호 |
| `RTE_NM` | 노선명 |
| `STOPS_ID` | 표준버스정류장ID |
| `STOPS_ARS_NO` | 버스정류장ARS번호 |
| `SBWY_STNS_NM` | 역명 |
| `TRFC_MNS_TYPE_CD` | 교통수단타입코드 |
| `TRFC_MNS_TYPE_NM` | 교통수단타입명 |
| `REG_YMD` | 등록일자 |

### C. 시간대 컬럼 처리 규약

| 데이터셋 | 영문 패턴 | 한글 패턴 |
|---|---|---|
| Subway | `HR_n_GET_ON/OFF_NOPE` | `NN시-NN시 승차/하차인원` |
| Bus | `HR_n_GET_ON/OFF_TNOPE` (HR_1만 `NOPE`) | `n시 승차/하차총승객수` |

**Bus HR_1 inconsistency**: 정규식 `HR_\d+_GET_(ON|OFF)_T?NOPE`로 통합 처리.

### D. 다음 단계

1. 이 노트북을 로컬에서 실행 → Part 1·2 결과 캡처 (시간대 컬럼 형식이 사이트와 다르면 즉시 수정)
2. WORKFLOW.md [3-2] `preprocess_subway.py`, [3-3] `preprocess_bus.py` 작성
   - `subway_rename`, `bus_rename` dict를 그대로 import / 재사용
   - 인코딩 = UTF-8
   - wide → long 변환 시 한글 컬럼명 기준
3. Hive 테이블 정의 ([4]) — 컬럼명은 한글 사용 시 백틱(``)으로 감싸야 함을 유의
